In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure pad_token is set to eos_token to avoid padding errors
mistral_tokenizer.pad_token = mistral_tokenizer.eos_token
xlstm_tokenizer.pad_token = xlstm_tokenizer.eos_token

# ---------------------------------------------------------
# 1. MISTRAL 7B ATTENTION EXTRACTION
# ---------------------------------------------------------
mistral_inputs = mistral_tokenizer(sample_text, return_tensors="pt").to(device)
with torch.no_grad():
    mistral_outputs = mistral(**mistral_inputs, output_attentions=True)

# Extract attention matrix from middle layer (e.g., layer 15)
# attentions shape: (batch_size, num_heads, sequence_length, sequence_length)
mistral_attn = mistral_outputs.attentions[15]
mistral_attn_map = mistral_attn[0].mean(dim=0).cpu().numpy() # Average across heads
mistral_tokens = mistral_tokenizer.convert_ids_to_tokens(mistral_inputs['input_ids'][0])

# ---------------------------------------------------------
# 2. xLSTM PSEUDO-ATTENTION EXTRACTION (Linear Attention Duality)
# ---------------------------------------------------------
# Prepare xLSTM inputs by prepending BOS token (as seen in logit_lens.ipynb)
xlstm_input_ids = xlstm_tokenizer(sample_text, return_tensors="pt")['input_ids'].to(device)
bos_id = xlstm_tokenizer.bos_token_id
if bos_id is not None:
    bos_tensor = torch.tensor([[bos_id]], device=device, dtype=xlstm_input_ids.dtype)
    xlstm_input_ids = torch.cat([bos_tensor, xlstm_input_ids], dim=1)
xlstm_tokens = xlstm_tokenizer.convert_ids_to_tokens(xlstm_input_ids[0])

# Dictionary to capture the internal tensors
captured_states = {}
def get_hook(name):
    def hook(module, input, output):
        captured_states[name] = output.detach()
    return hook

# Register forward hooks on the middle mLSTM layer (layer 15)
layer_idx = 15
mlstm = xlstm.backbone.blocks[layer_idx].mlstm_layer

h_q = mlstm.q.register_forward_hook(get_hook('q'))
h_k = mlstm.k.register_forward_hook(get_hook('k'))
h_i = mlstm.igate_preact.register_forward_hook(get_hook('igate_preact'))
h_f = mlstm.fgate_preact.register_forward_hook(get_hook('fgate_preact'))

# Forward pass to trigger hooks
with torch.no_grad():
    _ = xlstm(xlstm_input_ids)

# Remove hooks to keep the model clean
h_q.remove()
h_k.remove()
h_i.remove()
h_f.remove()

# Retrieve the captured tensors (squeeze the batch dimension)
q = captured_states['q'][0] # shape: (T, 2048)
k = captured_states['k'][0] # shape: (T, 2048)
igate = captured_states['igate_preact'][0] # shape: (T, 8)
fgate = captured_states['fgate_preact'][0] # shape: (T, 8)

T = q.shape[0]
num_heads = 8
head_dim = q.shape[-1] // num_heads # 256

# Reshape into (T, num_heads, head_dim) to process head-by-head
q = q.view(T, num_heads, head_dim)
k = k.view(T, num_heads, head_dim)

xlstm_attn_map = torch.zeros(T, T, device=device)

# Compute "Linear Attention Duality" manually
for h in range(num_heads):
    q_h = q[:, h, :] # (T, 256)
    k_h = k[:, h, :] # (T, 256)
    igate_h = igate[:, h] # (T,)
    fgate_h = fgate[:, h] # (T,)
    
    # Exponential input and forget gates
    i_tilde = torch.exp(igate_h)
    f = torch.exp(fgate_h)
    
    # Compute normalizer state (n) over sequence
    n = torch.zeros(T, head_dim, device=device)
    n_prev = torch.zeros(head_dim, device=device)
    for t in range(T):
        n_t = f[t] * n_prev + i_tilde[t] * k_h[t]
        n[t] = n_t
        n_prev = n_t
        
    # Construct the T x T pseudo-attention matrix with Causal Masking
    alpha_h = torch.zeros(T, T, device=device)
    for t in range(T):
        for i in range(t + 1): # Causal masking: i <= t
            if i == t:
                f_prod = 1.0
            else:
                f_prod = torch.prod(f[i+1:t+1])
            
            # As requested: alpha[t, i] = ( exp(dot(q_t, k_i)) * i_tilde_i * prod(f_{j} from j=i+1 to t) ) / dot(n_t, q_t)
            num = torch.exp(torch.dot(q_h[t], k_h[i])) * i_tilde[i] * f_prod
            den = torch.dot(n[t], q_h[t])
            
            if den != 0:
                alpha_h[t, i] = num / den
                
    xlstm_attn_map += alpha_h

# Average across heads
xlstm_attn_map = (xlstm_attn_map / num_heads).cpu().numpy()

# Clean up tokens for better visualization (optional but helps legibility)
mistral_tokens_clean = [t.replace(' ', '') for t in mistral_tokens]
xlstm_tokens_clean = [t.replace('Ġ', '').replace(' ', '') for t in xlstm_tokens]

# ---------------------------------------------------------
# 3. VISUALIZATION
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left Subplot: Mistral 7B
sns.heatmap(mistral_attn_map, xticklabels=mistral_tokens_clean, yticklabels=mistral_tokens_clean, cmap='viridis', ax=axes[0])
axes[0].set_title("Mistral 7B Attention Map (Layer 15)")
axes[0].set_xlabel("Key Tokens")
axes[0].set_ylabel("Query Tokens")
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Right Subplot: xLSTM
sns.heatmap(xlstm_attn_map, xticklabels=xlstm_tokens_clean, yticklabels=xlstm_tokens_clean, cmap='viridis', ax=axes[1])
axes[1].set_title("xLSTM Pseudo-Attention Map (Layer 15)")
axes[1].set_xlabel("Key Tokens")
axes[1].set_ylabel("Query Tokens")
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure pad_token is set to eos_token to avoid padding errors
mistral_tokenizer.pad_token = mistral_tokenizer.eos_token
xlstm_tokenizer.pad_token = xlstm_tokenizer.eos_token

# ---------------------------------------------------------
# 1. MISTRAL 7B ATTENTION EXTRACTION
# ---------------------------------------------------------
mistral_inputs = mistral_tokenizer(sample_text, return_tensors="pt").to(device)
with torch.no_grad():
    mistral_outputs = mistral(**mistral_inputs, output_attentions=True)

# Extract attention matrix from middle layer (e.g., layer 15)
# attentions shape: (batch_size, num_heads, sequence_length, sequence_length)
mistral_attn = mistral_outputs.attentions[15]
mistral_attn_map = mistral_attn[0].mean(dim=0).cpu().numpy() # Average across heads
mistral_tokens = mistral_tokenizer.convert_ids_to_tokens(mistral_inputs['input_ids'][0])

# ---------------------------------------------------------
# 2. xLSTM PSEUDO-ATTENTION EXTRACTION (Linear Attention Duality)
# ---------------------------------------------------------
# Prepare xLSTM inputs by prepending BOS token (as seen in logit_lens.ipynb)
xlstm_input_ids = xlstm_tokenizer(sample_text, return_tensors="pt")['input_ids'].to(device)
bos_id = xlstm_tokenizer.bos_token_id
if bos_id is not None:
    bos_tensor = torch.tensor([[bos_id]], device=device, dtype=xlstm_input_ids.dtype)
    xlstm_input_ids = torch.cat([bos_tensor, xlstm_input_ids], dim=1)
xlstm_tokens = xlstm_tokenizer.convert_ids_to_tokens(xlstm_input_ids[0])

# Dictionary to capture the internal tensors
captured_states = {}
def get_hook(name):
    def hook(module, input, output):
        captured_states[name] = output.detach()
    return hook

# Register forward hooks on the middle mLSTM layer (layer 15)
layer_idx = 15
mlstm = xlstm.backbone.blocks[layer_idx].mlstm_layer

h_q = mlstm.q.register_forward_hook(get_hook('q'))
h_k = mlstm.k.register_forward_hook(get_hook('k'))
h_i = mlstm.igate_preact.register_forward_hook(get_hook('igate_preact'))
h_f = mlstm.fgate_preact.register_forward_hook(get_hook('fgate_preact'))

# Forward pass to trigger hooks
with torch.no_grad():
    _ = xlstm(xlstm_input_ids)

# Remove hooks to keep the model clean
h_q.remove()
h_k.remove()
h_i.remove()
h_f.remove()

# Retrieve the captured tensors (squeeze the batch dimension)
q = captured_states['q'][0] # shape: (T, 2048)
k = captured_states['k'][0] # shape: (T, 2048)
igate = captured_states['igate_preact'][0] # shape: (T, 8)
fgate = captured_states['fgate_preact'][0] # shape: (T, 8)

T = q.shape[0]
num_heads = 8
head_dim = q.shape[-1] // num_heads # 256

# Reshape into (T, num_heads, head_dim) to process head-by-head
q = q.view(T, num_heads, head_dim)
k = k.view(T, num_heads, head_dim)

xlstm_attn_map = torch.zeros(T, T, device=device)

# Compute "Linear Attention Duality" manually
for h in range(num_heads):
    q_h = q[:, h, :] # (T, 256)
    k_h = k[:, h, :] # (T, 256)
    igate_h = igate[:, h] # (T,)
    fgate_h = fgate[:, h] # (T,)
    
    # Exponential input and forget gates
    i_tilde = torch.exp(igate_h)
    f = torch.exp(fgate_h)
    
    # Compute normalizer state (n) over sequence
    n = torch.zeros(T, head_dim, device=device)
    n_prev = torch.zeros(head_dim, device=device)
    for t in range(T):
        n_t = f[t] * n_prev + i_tilde[t] * k_h[t]
        n[t] = n_t
        n_prev = n_t
        
    # Construct the T x T pseudo-attention matrix with Causal Masking
    alpha_h = torch.zeros(T, T, device=device)
    for t in range(T):
        for i in range(t + 1): # Causal masking: i <= t
            if i == t:
                f_prod = 1.0
            else:
                f_prod = torch.prod(f[i+1:t+1])
            
            # As requested: alpha[t, i] = ( exp(dot(q_t, k_i)) * i_tilde_i * prod(f_{j} from j=i+1 to t) ) / dot(n_t, q_t)
            num = torch.exp(torch.dot(q_h[t], k_h[i])) * i_tilde[i] * f_prod
            den = torch.dot(n[t], q_h[t])
            
            if den != 0:
                alpha_h[t, i] = num / den
                
    xlstm_attn_map += alpha_h

# Average across heads
xlstm_attn_map = (xlstm_attn_map / num_heads).cpu().numpy()

# Clean up tokens for better visualization (optional but helps legibility)
mistral_tokens_clean = [t.replace(' ', '') for t in mistral_tokens]
xlstm_tokens_clean = [t.replace('Ġ', '').replace(' ', '') for t in xlstm_tokens]

# ---------------------------------------------------------
# 3. VISUALIZATION
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left Subplot: Mistral 7B
sns.heatmap(mistral_attn_map, xticklabels=mistral_tokens_clean, yticklabels=mistral_tokens_clean, cmap='viridis', ax=axes[0])
axes[0].set_title("Mistral 7B Attention Map (Layer 15)")
axes[0].set_xlabel("Key Tokens")
axes[0].set_ylabel("Query Tokens")
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Right Subplot: xLSTM
sns.heatmap(xlstm_attn_map, xticklabels=xlstm_tokens_clean, yticklabels=xlstm_tokens_clean, cmap='viridis', ax=axes[1])
axes[1].set_title("xLSTM Pseudo-Attention Map (Layer 15)")
axes[1].set_xlabel("Key Tokens")
axes[1].set_ylabel("Query Tokens")
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample_text = "The quick brown fox jumps over the lazy dog."

# ---------------------------------------------------------
# 1. MISTRAL 7B ATTENTION EXTRACTION
# ---------------------------------------------------------
mistral_inputs = mistral_tokenizer(sample_text, return_tensors="pt").to(device)
with torch.no_grad():
    mistral_outputs = mistral(**mistral_inputs, output_attentions=True)

# Extract attention matrix from middle layer (e.g., layer 15)
# attentions shape: (batch_size, num_heads, sequence_length, sequence_length)
mistral_attn = mistral_outputs.attentions[15]
mistral_attn_map = mistral_attn[0].mean(dim=0).cpu().numpy() # Average across heads
mistral_tokens = mistral_tokenizer.convert_ids_to_tokens(mistral_inputs['input_ids'][0])

# ---------------------------------------------------------
# 2. xLSTM PSEUDO-ATTENTION EXTRACTION (Linear Attention Duality)
# ---------------------------------------------------------
# Prepare xLSTM inputs by prepending BOS token (as seen in logit_lens.ipynb)
xlstm_input_ids = xlstm_tokenizer(sample_text, return_tensors="pt")['input_ids'].to(device)
bos_id = xlstm_tokenizer.bos_token_id
if bos_id is not None:
    bos_tensor = torch.tensor([[bos_id]], device=device, dtype=xlstm_input_ids.dtype)
    xlstm_input_ids = torch.cat([bos_tensor, xlstm_input_ids], dim=1)
xlstm_tokens = xlstm_tokenizer.convert_ids_to_tokens(xlstm_input_ids[0])

# Dictionary to capture the internal tensors
captured_states = {}
def get_hook(name):
    def hook(module, input, output):
        captured_states[name] = output.detach()
    return hook

# Register forward hooks on the middle mLSTM layer (layer 15)
layer_idx = 15
mlstm = xlstm.backbone.blocks[layer_idx].mlstm_layer

h_q = mlstm.q.register_forward_hook(get_hook('q'))
h_k = mlstm.k.register_forward_hook(get_hook('k'))
h_i = mlstm.igate_preact.register_forward_hook(get_hook('igate_preact'))
h_f = mlstm.fgate_preact.register_forward_hook(get_hook('fgate_preact'))

# Forward pass to trigger hooks
with torch.no_grad():
    _ = xlstm(xlstm_input_ids)

# Remove hooks to keep the model clean
h_q.remove()
h_k.remove()
h_i.remove()
h_f.remove()

# Retrieve the captured tensors (squeeze the batch dimension)
q = captured_states['q'][0] # shape: (T, 2048)
k = captured_states['k'][0] # shape: (T, 2048)
igate = captured_states['igate_preact'][0] # shape: (T, 8)
fgate = captured_states['fgate_preact'][0] # shape: (T, 8)

T = q.shape[0]
num_heads = 8
head_dim = q.shape[-1] // num_heads # 256

# Reshape into (T, num_heads, head_dim) to process head-by-head
q = q.view(T, num_heads, head_dim)
k = k.view(T, num_heads, head_dim)

xlstm_attn_map = torch.zeros(T, T, device=device)

# Compute "Linear Attention Duality" manually
for h in range(num_heads):
    q_h = q[:, h, :] # (T, 256)
    k_h = k[:, h, :] # (T, 256)
    igate_h = igate[:, h] # (T,)
    fgate_h = fgate[:, h] # (T,)
    
    # Exponential input and forget gates
    i_tilde = torch.exp(igate_h)
    f = torch.exp(fgate_h)
    
    # Compute normalizer state (n) over sequence
    n = torch.zeros(T, head_dim, device=device)
    n_prev = torch.zeros(head_dim, device=device)
    for t in range(T):
        n_t = f[t] * n_prev + i_tilde[t] * k_h[t]
        n[t] = n_t
        n_prev = n_t
        
    # Construct the T x T pseudo-attention matrix with Causal Masking
    alpha_h = torch.zeros(T, T, device=device)
    for t in range(T):
        for i in range(t + 1): # Causal masking: i <= t
            if i == t:
                f_prod = 1.0
            else:
                f_prod = torch.prod(f[i+1:t+1])
            
            # As requested: alpha[t, i] = ( exp(dot(q_t, k_i)) * i_tilde_i * prod(f_{j} from j=i+1 to t) ) / dot(n_t, q_t)
            num = torch.exp(torch.dot(q_h[t], k_h[i])) * i_tilde[i] * f_prod
            den = torch.dot(n[t], q_h[t])
            
            if den != 0:
                alpha_h[t, i] = num / den
                
    xlstm_attn_map += alpha_h

# Average across heads
xlstm_attn_map = (xlstm_attn_map / num_heads).cpu().numpy()

# Clean up tokens for better visualization (optional but helps legibility)
mistral_tokens_clean = [t.replace(' ', '') for t in mistral_tokens]
xlstm_tokens_clean = [t.replace('Ġ', '').replace(' ', '') for t in xlstm_tokens]

# ---------------------------------------------------------
# 3. VISUALIZATION
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left Subplot: Mistral 7B
sns.heatmap(mistral_attn_map, xticklabels=mistral_tokens_clean, yticklabels=mistral_tokens_clean, cmap='viridis', ax=axes[0])
axes[0].set_title("Mistral 7B Attention Map (Layer 15)")
axes[0].set_xlabel("Key Tokens")
axes[0].set_ylabel("Query Tokens")
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Right Subplot: xLSTM
sns.heatmap(xlstm_attn_map, xticklabels=xlstm_tokens_clean, yticklabels=xlstm_tokens_clean, cmap='viridis', ax=axes[1])
axes[1].set_title("xLSTM Pseudo-Attention Map (Layer 15)")
axes[1].set_xlabel("Key Tokens")
axes[1].set_ylabel("Query Tokens")
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()
